In [18]:
# Standard Python
import os
import shutil
from pathlib import Path

# PDF Loading
from langchain_community.document_loaders import PyPDFLoader

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_community.vectorstores import Chroma

# Vector Database
import chromadb

# Local LLM
import ollama

import time

In [47]:
PDF_PATHS = [
    "10k_Microsoft.pdf",
    "10K_Nvidia.pdf",
    "10K_Apple.pdf",
    "10K_Alphabet.pdf"
]

all_pages = []

for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    for page in pages:
        page.metadata["doc_name"] = Path(pdf_path).name

    all_pages.extend(pages)

print(f"Loaded {len(all_pages)} total pages from {len(PDF_PATHS)} documents.")

Loaded 430 total pages from 4 documents.


In [33]:
SECTION_HEADINGS = [
    # Front Matter
    "Table of Contents",
    "Forward-Looking Statements",

    # Part I
    "Business",
    "Overview",
    "Products and Services",
    "Business Segments",
    "Markets",
    "Competition",
    "Strategy",
    "Technology",
    "Artificial Intelligence",
    "Research and Development",
    "Human Capital",
    "Sustainability",
    "Environmental",
    "Cybersecurity",
    "Risk Factors",
    "Properties",
    "Legal Proceedings",
    "Mine Safety Disclosures",

    # Part II
    "Market for Registrant's Common Equity",
    "Selected Financial Data",
    "Management's Discussion and Analysis",
    "MD&A",
    "Results of Operations",
    "Liquidity and Capital Resources",
    "Critical Accounting Estimates",
    "Quantitative and Qualitative Disclosures About Market Risk",
    "Financial Statements",
    "Notes to Financial Statements",
    "Controls and Procedures",
    "Other Information",

    # Part III
    "Directors",
    "Executive Officers",
    "Corporate Governance",
    "Executive Compensation",
    "Security Ownership",
    "Related Party Transactions",
    "Principal Accountant Fees and Services",

    # Part IV
    "Exhibits",
    "Financial Statement Schedules",
    "Signatures",
]

In [34]:
def detect_section(text, current_section):
    text_lower = text.lower()

    for heading in SECTION_HEADINGS:
        if heading.lower() in text_lower:
            return heading

    return current_section

In [35]:
current_section_by_doc = {}

for page in all_pages:
    doc_name = page.metadata.get("doc_name", "Unknown document")
    current_section = current_section_by_doc.get(doc_name, "Unknown section")

    page_number = page.metadata.get("page", 0) + 1
    text = page.page_content

    current_section = detect_section(text, current_section)

    heading_path = current_section

    page.metadata["page"] = page_number
    page.metadata["section"] = current_section
    page.metadata["heading_path"] = heading_path
    page.metadata["clause_number"] = "N/A"

    current_section_by_doc[doc_name] = current_section

In [37]:
# Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=250,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(all_pages)

print(f"Created {len(chunks)} chunks from {len(PDF_PATHS)} documents.")

Created 1142 chunks from 4 documents.


In [38]:
# Create Chroma database
class LocalSentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=True).tolist()

    def embed_query(self, text):
        return self.model.encode([text]).tolist()[0]

In [39]:
CHROMA_PATH = f"./chroma_policy_db_{int(time.time())}"

def save_to_chroma(chunks):
    # Clear old database first
    if os.path.exists(CHROMA_PATH):
        shutil.rmtree(CHROMA_PATH)
        print("Deleted existing Chroma database.")

    embedding_function = LocalSentenceTransformerEmbeddings()

    db = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory=CHROMA_PATH
    )

    print(f"Saved {len(chunks)} chunks to {CHROMA_PATH}.")
    
    return db
    
save_to_chroma(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Saved 1142 chunks to ./chroma_policy_db_1784648010.


In [40]:
# Load dbase

embedding_function = LocalSentenceTransformerEmbeddings()

db = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embedding_function
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [41]:
# TESTING

question = "What are Microsoft's primary business segments?"

results = db.similarity_search_with_relevance_scores(
    question,
    k=5
)

for i, (doc, score) in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Score:", score)
    print("Metadata:", doc.metadata)
    print("Text preview:")
    print(doc.page_content[:800])
    print()

--- Result 1 ---
Score: 0.40153704682816704
Metadata: {'title': '0000320193-25-000079', 'keywords': '0000320193-25-000079; ; 10-K', 'page': 50, 'total_pages': 80, 'creator': 'EDGAR Filing HTML Converter', 'moddate': '2025-10-31T06:07:55-04:00', 'page_label': '50', 'section': 'Business', 'subject': 'Form 10-K filed on 2025-10-31 for the period ending 2025-09-27', 'doc_name': '10K_Apple.pdf', 'heading_path': 'Business', 'clause_number': 'N/A', 'source': '10K_Apple.pdf', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'creationdate': '2025-10-31T06:07:30-04:00', 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0'}
Text preview:
Note 13 – Segment Information and Geographic Data
The Company manages its business primarily on a geographic basis. The Company’s CEO is its CODM.
The Company’s reportable segments consist of the Americas, Europe, Greater China, Japan and Rest of Asia Pacific. Americas includes both North and South
America. Europe includes European countries, as wel

In [42]:
# Prompting

def build_prompt(question, results):
    context_blocks = []
    
    for i, (doc, score) in enumerate(results, start=1):
        meta = doc.metadata
    
        citation = (
            f'{meta.get("doc_name")} | '
            f'{meta.get("heading_path", meta.get("section"))} | '
            f'p.{meta.get("page")}'
        )
    
        context_blocks.append(
            f"[Source {i}: {citation}]\n{doc.page_content}"
        )
    
    context = "\n\n".join(context_blocks)
    
    prompt = f"""
    
    You are a Financial Report Intelligence Assistant.
    
    Your task is to answer questions ONLY using the information provided from SEC Form 10-K annual reports.
    
    The retrieved context may come from multiple companies and multiple sections of their reports.
    
    ------------------------
    RULES
    ------------------------
    
    1. Use ONLY the provided context.
       - Do NOT use outside knowledge.
       - Do NOT use prior knowledge about any company.
       - Do NOT guess or infer facts that are not explicitly stated.
    
    2. If multiple sources discuss the same topic:
       - Combine the information into one coherent answer.
       - Clearly distinguish companies when appropriate.
    
    3. If the question asks for a comparison:
       - Compare ONLY the information explicitly stated in the provided context.
       - Do not invent differences.
       - If one company does not discuss the topic, state that it is not mentioned in the retrieved context.
    
    4. If the answer cannot be determined from the retrieved context, respond EXACTLY with:
    
    I cannot find a definitive answer in the provided financial reports.
    
    Do not add any explanation.
    
    5. Never fabricate financial figures.
    
    6. Never estimate values.
    
    7. Never answer using general business knowledge.
    
    8. When mentioning companies, use their names consistently.
    
    ------------------------
    RESPONSE FORMAT
    ------------------------
    
    Answer:
    <your answer>
    
    Sources:
    - Source 1
    - Source 2
    
    Only include sources that directly support the answer.
    
    ------------------------
    QUESTION
    ------------------------
    
    {question}
    
    ------------------------
    FINANCIAL REPORT CONTEXT
    ------------------------
    
    {context}
    
    ------------------------
    ANSWER
    ------------------------
    """

    return prompt

In [43]:
NO_ANSWER = "I cannot find a definitive answer in the provided policy wording."

def generate_answer(question, k=5, model_name="llama3"):
    results = db.similarity_search_with_relevance_scores(
        question,
        k=k
    )

    prompt = build_prompt(question, results)

    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    answer = response["message"]["content"].strip()

    if NO_ANSWER in answer:
        return NO_ANSWER, results

    return answer, results

In [45]:
# AGENT

answer, results = generate_answer(
    "What are Microsoft's primary business segments?",
    k=5,
    model_name="llama3"
)

print(answer)

Answer:
Microsoft's primary business segments are Productivity and Business Processes, Intelligent Cloud, and More Personal Computing.

Sources:

* [Source 3: 10k_Microsoft.pdf | Business | p.5] PART IItem 1 
* [Source 4: 10k_Microsoft.pdf | Business | p.82] NOTE 18 — SEGMENT INFORMATION AND GEOGRAPHIC DATA
* [Source 5: 10k_Microsoft.pdf | Business | p.38] PART IIItem 7


In [46]:
test_questions = [
    # In-Domain (Should Answer)
    "What are Microsoft's primary business segments?",
    "Summarize Apple's main risk factors.",
    "What are Alphabet's key artificial intelligence initiatives?",
    "How does Amazon describe its cloud computing business?",
    "What cybersecurity risks are discussed in Microsoft's annual report?",

    # Near-Miss (Should Return No Definitive Answer)
    "Which company has the best long-term investment potential?",
    "Will Microsoft's AI investments increase its stock price?",
    "Which company will generate the highest revenue next year?",

    # Out-of-Scope (Should Return No Definitive Answer)
    "Should I buy Microsoft or Apple stock?",
    "Predict which company will have the highest revenue in 2030."
]

for q in test_questions:
    answer, _ = generate_answer(q)
    print("\nQUESTION:", q)
    print(answer)


QUESTION: What are Microsoft's primary business segments?
Answer:
Microsoft's primary business segments are:

* Productivity and Business Processes, which includes products and services such as Microsoft 365 Commercial cloud, Microsoft 365 Commercial products, Windows Commercial on-premises, Office licensed on-premises, and others.
* Intelligent Cloud, which includes Azure, Microsoft's cloud computing platform, as well as other cloud-based services and solutions.
* More Personal Computing, which includes the company's consumer-focused businesses, such as Windows OEM and Devices (including Surface), Xbox content and services, search and news advertising, and other consumer-facing products and services.

Sources:
- Source 1: Note 13 – Segment Information and Geographic Data from Apple's 10K report
- Source 2: Microsoft's Business segment information from their 10K report
- Source 3: Microsoft's Operating Segments from their 10K report
- Source 4: Note 18 — Segment Information and Geogra